# 02 — RFM Segmentation
**Key rule:** Customer IDs are stored as **raw hex strings** throughout this notebook.  
No hashing. No integer conversion. This is what allows notebook 03 to merge correctly.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

## 1. Load Data — Raw IDs Only

In [ ]:
import pandas as pd

# Define the columns strictly needed for RFM (Recency, Frequency, Monetary)
# and for joining with demographic data later.
trans_cols = ['t_dat', 'customer_id', 'price'] 

transactions = pd.read_csv(
    DATA_RAW / 'transactions_train.csv',
    usecols=trans_cols,
    dtype={
        'customer_id': 'str', # Read as string first to avoid casting issues
        'price': 'float32'
    },
    parse_dates=['t_dat'],
    engine='c' # Use 'c' instead of 'pyarrow' if pyarrow keeps failing realloc
)

# IMMEDIATELY convert to category to save 80% RAM
transactions['customer_id'] = transactions['customer_id'].astype('category')

In [ ]:
from pathlib import Path

# Set your base directory (adjust this if your notebook is in a subfolder)
BASE_DIR = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

# Define the paths based on your image structure
DATA_RAW = BASE_DIR / 'data' / 'raw'
PROCESSED_DATA = BASE_DIR / 'data' / 'processed' 

# Create the directory if it doesn't exist (good practice for your pipeline)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the cleaned customer data created in Notebook 01
# Use the optimized dtypes we discussed to save memory
customers = pd.read_csv(
PROCESSED_DATA/ 'customers_clean.csv', 
    dtype={
        'customer_id': 'str',
        'club_member_status': 'category',
        'fashion_news_frequency': 'category'
    }
)

# Crucial: Ensure customer_id is a category in BOTH for a fast merge
customers['customer_id'] = customers['customer_id'].astype('category')
rfm['customer_id'] = rfm['customer_id'].astype('category')

## 2. Compute RFM Features

In [ ]:
SNAPSHOT_DATE = transactions['t_dat'].max() + pd.Timedelta(days=1)
print(f'Snapshot date (1 day after last TX): {SNAPSHOT_DATE.date()}')

rfm = (
    transactions
    .groupby('customer_id')
    .agg(
        recency   = ('t_dat',   lambda x: (SNAPSHOT_DATE - x.max()).days),
        frequency = ('t_dat',   'count'),
        monetary  = ('price',   'sum')
    )
    .reset_index()
)

print(f'RFM shape: {rfm.shape}')
rfm.describe()

## 3. Merge with Customer Attributes

In [ ]:
customer_cols = [
    'customer_id', 'FN', 'Active', 'club_member_status',
    'fashion_news_frequency', 'age', 'age_group',
    'engagement_score', 'age_bucket'
]
# keep only columns that actually exist
customer_cols = [c for c in customer_cols if c in customers.columns]

rfm = rfm.merge(customers[customer_cols], on='customer_id', how='left')
print(f'After merge: {rfm.shape}')
print('Null counts:')
print(rfm.isna().sum())

## 4. KMeans Clustering on RFM

In [ ]:
N_CLUSTERS = 5
RANDOM_STATE = 42

features = rfm[['recency', 'frequency', 'monetary']].copy()

# Log-transform monetary and frequency to reduce skew
features['frequency'] = np.log1p(features['frequency'])
features['monetary']  = np.log1p(features['monetary'])

# Fill any remaining nulls
features = features.fillna(features.median())

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
rfm['cluster'] = kmeans.fit_predict(X_scaled)

print('Cluster distribution:')
print(rfm['cluster'].value_counts().sort_index())

## 5. Sanity Check — Cluster Profiles

In [ ]:
profile = rfm.groupby('cluster')[['recency', 'frequency', 'monetary']].mean().round(2)
print(profile)

# Verify customer_id is still a raw hex string — NOT a hashed integer
sample = rfm['customer_id'].iloc[0]
assert len(sample) > 20 and not sample.lstrip('-').isdigit(), (
    f'❌ customer_id looks like a hash ({sample!r}). '
    'Check whether customers_clean.csv still has raw IDs.'
)
print(f'\n customer_id format OK: {sample[:30]}...')

## 6. Save rfm_segmented.csv

In [ ]:
rfm.to_csv(DATA_PROC / 'rfm_segmented.csv', index=False)
print(f'Saved rfm_segmented.csv  — shape {rfm.shape}')
print(f'   Sample customer_id: {rfm["customer_id"].iloc[0]}')
print(f'   Columns: {rfm.columns.tolist()}')
rfm.head(3)

In [ ]:
# Save the final RFM and Segmented data
# This will be used by your Streamlit app or Notebook 03
rfm.to_parquet(PROCESSED_DATA / 'rfm_segmented.parquet', index=False)

# If you have other key dataframes, save them too
# customers.to_parquet(PROCESSED_DATA / 'customers_clean.parquet', index=False)

print("All files saved successfully to:", PROCESSED_DATA)